In [5]:
pip install opencv-python ultralytics

Defaulting to user installation because normal site-packages is not writeable
  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached torchvision-0.28.0-cp314-cp314-win_amd64.whl.metadata (5.6 kB)
  Using cached polars-1.44.1-py3-none-any.whl.metadata (11 kB)
  Using cached nvidia_ml_py-13.610.43-py3-none-any.whl.metadata (9.7 kB)
  Using cached ultralytics_thop-2.1.6-py3-none-any.whl.metadata (13 kB)
  Using cached ultralytics_platform-0.1.21-py3-none-any.whl.metadata (8.4 kB)
  Using cached polars_runtime_32-1.44.1-cp310-abi3-win_amd64.whl.metadata (1.5 kB)
  Using cached torch-2.13.0-cp314-cp314-win_amd64.whl.metadata (39 kB)
Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl (44.0 MB)
   ---------------------------------------- 0.0/1.4 MB ? eta -:--:--
   ----------------------------- ---------- 1.0/1.4 MB 10.1 MB/s eta 0:00:01
   ---------------------------------------- 1.4/1.4 MB 8.2 MB/s  0:00:00
Using cached nvidia_ml_py-13.610.43-

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# downloading the video  and diving it into frames and identifying the object present in the frame and finally storing the extracted frames 

In [11]:
import os
import cv2
from ultralytics import YOLO


class VideoObjectDetector:

    def __init__(
        self,
        video_path,
        output_folder="extracted_frames",
        frame_interval=30
    ):
        self.video_path = video_path
        self.output_folder = output_folder
        self.frame_interval = frame_interval

        self.model = None
        self.cap = None
        self.frame_count = 0
        self.saved_count = 0

    def create_output_folder(self):

        # Create output folder
        os.makedirs(
            self.output_folder,
            exist_ok=True
        )

    def load_model(self):

        # Load YOLO model
        print("\nLoading YOLO model...")

        self.model = YOLO("yolov8n.pt")

        print("Model loaded successfully.")

    def open_video(self):

        # Open the video file
        self.cap = cv2.VideoCapture(
            self.video_path
        )

        if not self.cap.isOpened():
            print(
                f"Error: Could not open video file "
                f"{self.video_path}"
            )
            return False

        print("Video opened successfully.")
        return True

    def process_frame(self, frame):

        # Detect objects in the frame
        results = self.model(
            frame,
            verbose=False
        )

        detected_objects = []

        for result in results:

            for box in result.boxes:

                class_id = int(
                    box.cls[0]
                )

                object_name = self.model.names[
                    class_id
                ]

                detected_objects.append(
                    object_name
                )

        unique_objects = list(
            set(detected_objects)
        )

        if unique_objects:
            objects_text = ", ".join(
                unique_objects
            )
        else:
            objects_text = "None"

        # Draw bounding boxes
        annotated_frame = results[0].plot()

        return annotated_frame, objects_text

    def save_frame(
        self,
        annotated_frame,
        objects_text
    ):

        # Save the processed frame
        frame_filename = os.path.join(
            self.output_folder,
            f"frame_{self.saved_count:04d}.jpg"
        )

        cv2.imwrite(
            frame_filename,
            annotated_frame
        )

        print(
            f"Saved: {frame_filename} | "
            f"Objects found: {objects_text}"
        )

        self.saved_count += 1

    def process_video(self):

        # Process the video
        print("\nProcessing video... Please wait.")

        while True:

            ret, frame = self.cap.read()

            if not ret:
                break

            if (
                self.frame_count
                % self.frame_interval
                == 0
            ):

                annotated_frame, objects_text = (
                    self.process_frame(frame)
                )

                self.save_frame(
                    annotated_frame,
                    objects_text
                )

            self.frame_count += 1

    def release_resources(self):

        # Release video resources
        if self.cap is not None:
            self.cap.release()

        print(
            f"\nTask Finished! Extracted "
            f"{self.saved_count} frames to the "
            f"'{self.output_folder}' folder."
        )

    def run(self):

        # Run the complete process
        self.create_output_folder()
        self.load_model()

        if not self.open_video():
            return

        self.process_video()
        self.release_resources()


if __name__ == "__main__":

    VIDEO_PATH = (
        r"C:\Users\KISHORI KAMALI"
        r"\OneDrive\Desktop"
        r"\56310-479197605_medium.mp4"
    )

    detector = VideoObjectDetector(
        video_path=VIDEO_PATH,
        output_folder="extracted_frames",
        frame_interval=30
    )

    detector.run()


Loading YOLO model...
Model loaded successfully.
Video opened successfully.

Processing video... Please wait.
Saved: extracted_frames\frame_0000.jpg | Objects found: truck, car
Saved: extracted_frames\frame_0001.jpg | Objects found: truck, car
Saved: extracted_frames\frame_0002.jpg | Objects found: truck, car
Saved: extracted_frames\frame_0003.jpg | Objects found: car
Saved: extracted_frames\frame_0004.jpg | Objects found: truck, car
Saved: extracted_frames\frame_0005.jpg | Objects found: truck, car
Saved: extracted_frames\frame_0006.jpg | Objects found: truck, bus, car
Saved: extracted_frames\frame_0007.jpg | Objects found: truck, bus, car
Saved: extracted_frames\frame_0008.jpg | Objects found: truck, bus, car
Saved: extracted_frames\frame_0009.jpg | Objects found: truck, car
Saved: extracted_frames\frame_0010.jpg | Objects found: truck, bus, car
Saved: extracted_frames\frame_0011.jpg | Objects found: truck, car
Saved: extracted_frames\frame_0012.jpg | Objects found: truck, car
Saved

# extracting the frames which has only trucks

In [12]:
import os
import cv2
from ultralytics import YOLO


class TargetObjectDetector:

    def __init__(
        self,
        video_path,
        target_object,
        output_folder="target_frames",
        frame_interval=30
    ):
        self.video_path = video_path
        self.target_object = target_object
        self.output_folder = output_folder
        self.frame_interval = frame_interval

        self.model = None
        self.cap = None
        self.frame_count = 0
        self.saved_count = 0
        self.matching_frames = []

    def create_output_folder(self):

        # Create output folder
        os.makedirs(
            self.output_folder,
            exist_ok=True
        )

    def load_model(self):

        # Load YOLO model
        self.model = YOLO(
            "yolov8n.pt"
        )

    def open_video(self):

        # Open the video file
        self.cap = cv2.VideoCapture(
            self.video_path
        )

        if not self.cap.isOpened():

            print(
                f"Error: Could not open video file "
                f"{self.video_path}"
            )

            return False

        return True

    def detect_target(self, frame):

        # Detect objects in the frame
        results = self.model(
            frame,
            verbose=False
        )

        target_found = False

        for result in results:

            for box in result.boxes:

                class_id = int(
                    box.cls[0]
                )

                object_name = self.model.names[
                    class_id
                ]

                if (
                    object_name.lower()
                    == self.target_object.lower()
                ):
                    target_found = True
                    break

            if target_found:
                break

        return results, target_found

    def save_matching_frame(self, results):

        # Save the matching frame
        frame_name = (
            f"frame_{self.saved_count:04d}.jpg"
        )

        self.matching_frames.append(
            frame_name
        )

        annotated_frame = results[0].plot()

        frame_filename = os.path.join(
            self.output_folder,
            frame_name
        )

        cv2.imwrite(
            frame_filename,
            annotated_frame
        )

        self.saved_count += 1

    def process_video(self):

        # Process the video
        print(
            f"Processing video to find all instances of: "
            f"'{self.target_object}'...\n"
        )

        while True:

            ret, frame = self.cap.read()

            if not ret:
                break

            if (
                self.frame_count
                % self.frame_interval
                == 0
            ):

                results, target_found = (
                    self.detect_target(frame)
                )

                if target_found:

                    self.save_matching_frame(
                        results
                    )

            self.frame_count += 1

    def release_video(self):

        # Release video resources
        if self.cap is not None:
            self.cap.release()

    def show_report(self):

        # Display final report
        print("=" * 50)
        print("TASK SUMMARY REPORT")
        print("=" * 50)

        print(
            f"Target Object searched: "
            f"{self.target_object}"
        )

        print(
            f"Total number of frames containing "
            f"'{self.target_object}': "
            f"{len(self.matching_frames)}"
        )

        print("-" * 50)

        if self.matching_frames:

            print(
                "List of frames where the object is present:"
            )

            for frame in self.matching_frames:

                print(
                    f" - {frame}"
                )

        else:

            print(
                "The target object was not detected "
                "in any processed frames."
            )

        print("=" * 50)

    def run(self):

        # Run the complete process
        self.create_output_folder()
        self.load_model()

        if not self.open_video():
            return

        self.process_video()
        self.release_video()
        self.show_report()


if __name__ == "__main__":

    VIDEO_PATH = (
        r"C:\Users\KISHORI KAMALI"
        r"\OneDrive\Desktop"
        r"\56310-479197605_medium.mp4"
    )

    TARGET_OBJECT = "truck"

    detector = TargetObjectDetector(
        video_path=VIDEO_PATH,
        target_object=TARGET_OBJECT,
        output_folder="target_frames",
        frame_interval=30
    )

    detector.run()

Processing video to find all instances of: 'truck'...

TASK SUMMARY REPORT
Target Object searched: truck
Total number of frames containing 'truck': 27
--------------------------------------------------
List of frames where the object is present:
 - frame_0000.jpg
 - frame_0001.jpg
 - frame_0002.jpg
 - frame_0003.jpg
 - frame_0004.jpg
 - frame_0005.jpg
 - frame_0006.jpg
 - frame_0007.jpg
 - frame_0008.jpg
 - frame_0009.jpg
 - frame_0010.jpg
 - frame_0011.jpg
 - frame_0012.jpg
 - frame_0013.jpg
 - frame_0014.jpg
 - frame_0015.jpg
 - frame_0016.jpg
 - frame_0017.jpg
 - frame_0018.jpg
 - frame_0019.jpg
 - frame_0020.jpg
 - frame_0021.jpg
 - frame_0022.jpg
 - frame_0023.jpg
 - frame_0024.jpg
 - frame_0025.jpg
 - frame_0026.jpg


# extracting frame that have red car 

In [13]:
import os
import cv2
import numpy as np
from ultralytics import YOLO


class RedCarDetector:

    def __init__(
        self,
        video_path,
        output_folder="extracted_red_cars",
        frame_interval=30
    ):
        self.video_path = video_path
        self.output_folder = output_folder
        self.frame_interval = frame_interval

        self.model = None
        self.cap = None
        self.frame_count = 0
        self.saved_count = 0

    def create_output_folder(self):

        # Create output folder
        os.makedirs(
            self.output_folder,
            exist_ok=True
        )

    def load_model(self):

        # Load YOLO model
        self.model = YOLO(
            "yolov8n.pt"
        )

    def open_video(self):

        # Open the video file
        self.cap = cv2.VideoCapture(
            self.video_path
        )

        if not self.cap.isOpened():

            print(
                f"Error: Could not open video file "
                f"{self.video_path}"
            )

            return False

        return True

    def check_red_color(self, car_crop):

        # Check the red color in the car
        if car_crop.size == 0:
            return False, 0

        hsv_crop = cv2.cvtColor(
            car_crop,
            cv2.COLOR_BGR2HSV
        )

        lower_red1 = np.array(
            [0, 70, 50]
        )

        upper_red1 = np.array(
            [10, 255, 255]
        )

        lower_red2 = np.array(
            [170, 70, 50]
        )

        upper_red2 = np.array(
            [180, 255, 255]
        )

        mask1 = cv2.inRange(
            hsv_crop,
            lower_red1,
            upper_red1
        )

        mask2 = cv2.inRange(
            hsv_crop,
            lower_red2,
            upper_red2
        )

        red_mask = mask1 + mask2

        red_pixel_ratio = (
            np.sum(red_mask > 0)
            / car_crop.size
        )

        return (
            red_pixel_ratio > 0.08,
            red_pixel_ratio
        )

    def detect_red_cars(self, frame):

        # Detect cars using YOLO
        results = self.model(
            frame,
            verbose=False
        )

        red_car_found = False
        annotated_frame = frame.copy()

        for result in results:

            for box in result.boxes:

                class_id = int(
                    box.cls[0]
                )

                object_name = self.model.names[
                    class_id
                ]

                if object_name == "car":

                    xyxy = (
                        box.xyxy
                        .cpu()
                        .numpy()[0]
                        .astype(int)
                    )

                    x1, y1, x2, y2 = xyxy

                    car_crop = frame[
                        y1:y2,
                        x1:x2
                    ]

                    is_red, red_pixel_ratio = (
                        self.check_red_color(
                            car_crop
                        )
                    )

                    if is_red:

                        red_car_found = True

                        cv2.rectangle(
                            annotated_frame,
                            (x1, y1),
                            (x2, y2),
                            (0, 0, 255),
                            3
                        )

                        cv2.putText(
                            annotated_frame,
                            f"Red Car {red_pixel_ratio:.2f}",
                            (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6,
                            (0, 0, 255),
                            2
                        )

        return (
            red_car_found,
            annotated_frame
        )

    def save_frame(self, annotated_frame):

        # Save the detected frame
        frame_filename = os.path.join(
            self.output_folder,
            f"frame_{self.saved_count:04d}.jpg"
        )

        cv2.imwrite(
            frame_filename,
            annotated_frame
        )

        print(
            f"Saved: {frame_filename} | "
            f"Video Frame Position: "
            f"{self.frame_count}"
        )

        self.saved_count += 1

    def process_video(self):

        # Process the video
        print(
            "Processing video for red cars... "
            "Please wait."
        )

        while True:

            ret, frame = self.cap.read()

            if not ret:
                break

            if (
                self.frame_count
                % self.frame_interval
                == 0
            ):

                red_car_found, annotated_frame = (
                    self.detect_red_cars(frame)
                )

                if red_car_found:

                    self.save_frame(
                        annotated_frame
                    )

            self.frame_count += 1

    def release_video(self):

        # Release video resources
        if self.cap is not None:
            self.cap.release()

    def show_result(self):

        print(
            f"\nTask Finished! Extracted "
            f"{self.saved_count} frames to the "
            f"'{self.output_folder}' folder."
        )

    def run(self):

        # Run the complete process
        self.create_output_folder()
        self.load_model()

        if not self.open_video():
            return

        self.process_video()
        self.release_video()
        self.show_result()


if __name__ == "__main__":

    VIDEO_PATH = (
        r"C:\Users\KISHORI KAMALI"
        r"\OneDrive\Desktop"
        r"\56310-479197605_medium.mp4"
    )

    detector = RedCarDetector(
        video_path=VIDEO_PATH,
        output_folder="extracted_red_cars",
        frame_interval=30
    )

    detector.run()

Processing video for red cars... Please wait.
Saved: extracted_red_cars\frame_0000.jpg | Video Frame Position: 390
Saved: extracted_red_cars\frame_0001.jpg | Video Frame Position: 420
Saved: extracted_red_cars\frame_0002.jpg | Video Frame Position: 450
Saved: extracted_red_cars\frame_0003.jpg | Video Frame Position: 780
Saved: extracted_red_cars\frame_0004.jpg | Video Frame Position: 810
Saved: extracted_red_cars\frame_0005.jpg | Video Frame Position: 840
Saved: extracted_red_cars\frame_0006.jpg | Video Frame Position: 870
Saved: extracted_red_cars\frame_0007.jpg | Video Frame Position: 900

Task Finished! Extracted 8 frames to the 'extracted_red_cars' folder.


In [9]:
!pip install pypdf sentence-transformers faiss-cpu numpy

Defaulting to user installation because normal site-packages is not writeable
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ---------------------------------------- 0.0/739.8 kB ? eta -:--:--
   ---------------------------------------- 739.8/739.8 kB 7.1 MB/s  0:00:00
   ---------------------------------------- 0.0/795.8 kB ? eta -:--:--
   ---------------------------------------- 795.8/795.8 kB 6.5 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------- ----------------------------- 1.0/4.0 MB 8.6 MB/s eta 0:00:01
   ----------------------- ---------------- 2.4/4.0 MB 6.9 MB/s eta 0:00:01
   ------------------------------- -------- 3.1/4.0 MB 6.4 MB/s eta 0:00:01
   -------------------------------------- - 3.9/4.0 MB 6.3 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 5.1 MB/s  0:00:00
   ---------------------------------------- 0.0/12.1 MB ? eta -:--:--
   --- ------------------------------------ 1.0/

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# task 1 extract the text divided into chunkes and converted as embedding using sentence transformer and also store it in faiss data structure 

In [9]:
import json
import faiss

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer


class DocumentProcessor:

    def __init__(
        self,
        pdf_path,
        faiss_index_path="document.index",
        chunks_path="chunks.json",
        chunk_size=150,
        chunk_overlap=50
    ):

        self.pdf_path = pdf_path
        self.faiss_index_path = faiss_index_path
        self.chunks_path = chunks_path
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

        self.text = ""
        self.chunks = []
        self.embeddings = None
        self.index = None
        self.model = None


    def load_model(self):

        # Load the Sentence Transformer model
        print("\nLoading Sentence Transformer model...")

        self.model = SentenceTransformer(
            "all-MiniLM-L6-v2"
        )

        print("Model loaded successfully.")


    def extract_text(self):

        # Extract text from all pages of the PDF
        print("\nExtracting PDF text...")

        reader = PdfReader(self.pdf_path)

        self.text = ""

        for page_number, page in enumerate(
            reader.pages,
            start=1
        ):

            page_text = page.extract_text()

            if page_text:
                self.text += page_text + "\n"

            print(f"Processed page {page_number}")

        print("Text extraction completed.")
        print("Characters extracted:", len(self.text))

        return self.text


    def create_chunks(self):

        # Divide the document text into overlapping chunks
        print("\nCreating document chunks...")

        words = self.text.split()

        self.chunks = []

        start = 0

        while start < len(words):

            end = start + self.chunk_size

            chunk = " ".join(
                words[start:end]
            )

            if chunk.strip():
                self.chunks.append(chunk)

            start += (
                self.chunk_size
                - self.chunk_overlap
            )

        print("Number of chunks:", len(self.chunks))

        return self.chunks


    def create_embeddings(self):

        # Convert each chunk into a numerical vector
        print("\nCreating embeddings...")

        self.embeddings = self.model.encode(
            self.chunks,
            convert_to_numpy=True,
            show_progress_bar=True
        )

        self.embeddings = self.embeddings.astype(
            "float32"
        )

        print("Embeddings created.")
        print("Embedding shape:", self.embeddings.shape)

        return self.embeddings


    def normalize_embeddings(self):

        # Normalize vectors for cosine similarity
        print("\nNormalizing embeddings...")

        faiss.normalize_L2(
            self.embeddings
        )

        print("Embeddings normalized.")


    def create_faiss_index(self):

        # Create the FAISS index and add the embeddings
        print("\nCreating FAISS index...")

        dimension = self.embeddings.shape[1]

        print("Embedding dimension:", dimension)

        self.index = faiss.IndexFlatIP(
            dimension
        )

        self.index.add(
            self.embeddings
        )

        print("Vectors added to FAISS.")
        print("Total vectors:", self.index.ntotal)


    def save_faiss_index(self):

        # Save the FAISS index to a file
        faiss.write_index(
            self.index,
            self.faiss_index_path
        )

        print(
            "FAISS index saved as:",
            self.faiss_index_path
        )


    def save_chunks(self):

        # Save the original chunks so they can be retrieved later
        with open(
            self.chunks_path,
            "w",
            encoding="utf-8"
        ) as file:

            json.dump(
                self.chunks,
                file,
                ensure_ascii=False,
                indent=2
            )

        print(
            "Chunks saved as:",
            self.chunks_path
        )


    def show_first_chunk(self):

        # Display one chunk as an example
        if len(self.chunks) > 0:

            print("\nExample chunk:")
            print("----------------------------------------")
            print(self.chunks[0])


    def process_document(self):

        # Run all document processing steps
        self.load_model()

        self.extract_text()

        self.create_chunks()

        self.show_first_chunk()

        self.create_embeddings()

        self.normalize_embeddings()

        self.create_faiss_index()

        self.save_faiss_index()

        self.save_chunks()

        print("\n========================================")
        print("TASK 1 COMPLETED SUCCESSFULLY")
        print("========================================")

        print("PDF:", self.pdf_path)
        print("Total chunks:", len(self.chunks))
        print(
            "Embedding dimension:",
            self.embeddings.shape[1]
        )
        print(
            "FAISS vectors:",
            self.index.ntotal
        )

        print("\nFiles created:")
        print("1.", self.faiss_index_path)
        print("2.", self.chunks_path)


if __name__ == "__main__":

    PDF_PATH = (
        r"C:\Users\KISHORI KAMALI"
        r"\Downloads\research.pdf"
    )

    processor = DocumentProcessor(
        pdf_path=PDF_PATH,
        faiss_index_path="document.index",
        chunks_path="chunks.json",
        chunk_size=150,
        chunk_overlap=50
    )

    processor.process_document()


Loading Sentence Transformer model...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2462.61it/s]


Model loaded successfully.

Extracting PDF text...
Processed page 1
Processed page 2
Processed page 3
Processed page 4
Text extraction completed.
Characters extracted: 5956

Creating document chunks...
Number of chunks: 9

Example chunk:
----------------------------------------
Document-Based Question Answering Using Sentence Transformers and FAISS 1. Introduction The proposed system is a document-based question answering system that uses Natural Language Processing (NLP), Sentence Transformer embeddings, and FAISS for semantic search. The system allows a user to upload a document, extracts the textual content, divides the content into smaller chunks, converts each chunk into a numerical vector representation, and stores the vectors in a FAISS index. When the user asks a question related to the document, the question is converted into the same vector representation and searched against the FAISS index. The system retrieves the most relevant document chunks and presents them as the answ

Batches: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.64it/s]

Embeddings created.
Embedding shape: (9, 384)

Normalizing embeddings...
Embeddings normalized.

Creating FAISS index...
Embedding dimension: 384
Vectors added to FAISS.
Total vectors: 9
FAISS index saved as: document.index
Chunks saved as: chunks.json

TASK 1 COMPLETED SUCCESSFULLY
PDF: C:\Users\KISHORI KAMALI\Downloads\research.pdf
Total chunks: 9
Embedding dimension: 384
FAISS vectors: 9

Files created:
1. document.index
2. chunks.json


# task 2 is to ask a question it should be coverted into embeddings and search into the faisss and return the relevant content with similarity score 

In [10]:
import json
import faiss
import re

from sentence_transformers import SentenceTransformer


class QuestionAnswerer:

    def __init__(
        self,
        faiss_index_path="document.index",
        chunks_path="chunks.json",
        top_k_chunks=3,
        top_k_sentences=3
    ):

        self.faiss_index_path = faiss_index_path
        self.chunks_path = chunks_path
        self.top_k_chunks = top_k_chunks
        self.top_k_sentences = top_k_sentences

        self.model = None
        self.index = None
        self.chunks = []
        self.question = ""
        self.question_embedding = None
        self.retrieved_chunks = []
        self.sentences = []


    def load_model(self):

        # Load the model used for text embeddings
        print("\nLoading Sentence Transformer model...")

        self.model = SentenceTransformer(
            "all-MiniLM-L6-v2"
        )

        print("Model loaded successfully.")


    def load_faiss_index(self):

        # Load the saved FAISS index
        print("\nLoading FAISS index...")

        self.index = faiss.read_index(
            self.faiss_index_path
        )

        print("FAISS index loaded.")
        print("Total vectors:", self.index.ntotal)


    def load_chunks(self):

        # Load the original text chunks
        print("\nLoading document chunks...")

        with open(
            self.chunks_path,
            "r",
            encoding="utf-8"
        ) as file:

            self.chunks = json.load(file)

        print("Chunks loaded:", len(self.chunks))


    def check_data(self):

        # Make sure the index and chunks have the same size
        if self.index.ntotal != len(self.chunks):

            print("\nWARNING!")
            print("FAISS vectors and chunks do not match.")

            print(
                "FAISS vectors:",
                self.index.ntotal
            )

            print(
                "Chunks:",
                len(self.chunks)
            )

        else:

            print(
                "FAISS vectors and chunks match correctly."
            )


    def get_question(self):

        
        print("DOCUMENT QUESTION ANSWERING")
        

        self.question = input(
            "\nEnter your question: "
        )


    def create_question_embedding(self):

        # Convert the question into an embedding
        self.question_embedding = self.model.encode(
            [self.question],
            convert_to_numpy=True
        )

        self.question_embedding = (
            self.question_embedding.astype("float32")
        )

        # Normalize the vector for cosine similarity
        faiss.normalize_L2(
            self.question_embedding
        )


    def search_faiss(self):

        # Find the most similar document chunks
        k = min(
            self.top_k_chunks,
            self.index.ntotal
        )

        distances, indices = self.index.search(
            self.question_embedding,
            k
        )

        self.retrieved_chunks = []

      
        print("RETRIEVED DOCUMENT CHUNKS")


        for rank, idx in enumerate(indices[0]):

            if idx < 0:
                continue

            if idx >= len(self.chunks):
                continue

            chunk = self.chunks[idx]

            self.retrieved_chunks.append(chunk)

            print(
                f"\nRelevant Chunk {rank + 1}"
            )

            print(
                "----------------------------------------"
            )

            print(chunk)

            print(
                "\nChunk Similarity Score:",
                round(
                    float(distances[0][rank]),
                    4
                )
            )


    def create_sentences(self):

        # Split the retrieved chunks into individual sentences
        self.sentences = []

        for chunk in self.retrieved_chunks:

            chunk_sentences = re.split(
                r'(?<=[.!?])\s+',
                chunk
            )

            for sentence in chunk_sentences:

                sentence = sentence.strip()

                if len(sentence) > 10:

                    self.sentences.append(
                        sentence
                    )


    def find_best_sentence(self):

        if len(self.sentences) == 0:

            print("\nNo valid sentences found.")

            return

        # Create embeddings for each sentence
        sentence_embeddings = self.model.encode(
            self.sentences,
            convert_to_numpy=True
        )

        sentence_embeddings = (
            sentence_embeddings.astype("float32")
        )

        # Normalize sentence vectors
        faiss.normalize_L2(
            sentence_embeddings
        )

        dimension = sentence_embeddings.shape[1]

        sentence_index = faiss.IndexFlatIP(
            dimension
        )

        sentence_index.add(
            sentence_embeddings
        )

        # Find the sentences closest to the question
        k = min(
            self.top_k_sentences,
            len(self.sentences)
        )

        distances, indices = sentence_index.search(
            self.question_embedding,
            k
        )

        # Give extra weight to words shared by the question and sentence
        question_words = set(
            re.findall(
                r'\b[a-zA-Z]+\b',
                self.question.lower()
            )
        )

        scored_sentences = []

        for rank, sentence_index_value in enumerate(indices[0]):

            sentence = self.sentences[
                sentence_index_value
            ]

            sentence_words = set(
                re.findall(
                    r'\b[a-zA-Z]+\b',
                    sentence.lower()
                )
            )

            common_words = (
                question_words & sentence_words
            )

            keyword_score = len(common_words)

            semantic_score = float(
                distances[0][rank]
            )

            final_score = (
                semantic_score
                + (keyword_score * 0.05)
            )

            scored_sentences.append(
                (
                    final_score,
                    semantic_score,
                    keyword_score,
                    sentence
                )
            )

        # Sort sentences by their combined score
        scored_sentences.sort(
            key=lambda x: x[0],
            reverse=True
        )

        best = scored_sentences[0]

   
        print("ANSWER")
      
        print("\n" + best[3])

        print(
            "\nSemantic Similarity:",
            round(best[1], 4)
        )

        print(
            "Keyword Matches:",
            best[2]
        )

        print(
            "Final Score:",
            round(best[0], 4)
        )

       
        print("OTHER RELEVANT SENTENCES")
       

        for rank, item in enumerate(
            scored_sentences[:self.top_k_sentences]
        ):

            print(
                f"\n{rank + 1}. {item[3]}"
            )

            print(
                "Semantic Similarity:",
                round(item[1], 4)
            )

            print(
                "Keyword Matches:",
                item[2]
            )

            print(
                "Final Score:",
                round(item[0], 4)
            )


    def answer_question(self):

        # Run the complete question-answering process
        self.load_model()
        self.load_faiss_index()
        self.load_chunks()
        self.check_data()
        self.get_question()
        self.create_question_embedding()
        self.search_faiss()
        self.create_sentences()
        self.find_best_sentence()

       
        print("TASK 2 COMPLETED")
       


if __name__ == "__main__":

    answerer = QuestionAnswerer(
        faiss_index_path="document.index",
        chunks_path="chunks.json",
        top_k_chunks=3,
        top_k_sentences=3
    )

    answerer.answer_question()


Loading Sentence Transformer model...


Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 2268.07it/s]


Model loaded successfully.

Loading FAISS index...
FAISS index loaded.
Total vectors: 9

Loading document chunks...
Chunks loaded: 9
FAISS vectors and chunks match correctly.
DOCUMENT QUESTION ANSWERING



Enter your question:  What is the main objective of the proposed system?


RETRIEVED DOCUMENT CHUNKS

Relevant Chunk 1
----------------------------------------
chunks and presents them as the answer. 2. Objective The main objective of this system is to develop an efficient semantic document search and question answering system. The system should: • Accept a document as input. • Extract textual content from the document. • Divide the extracted content into smaller chunks. • Generate embeddings for each chunk using a Sentence Transformer model. • Represent each chunk as a 384-dimensional vector. • Store the embeddings using the FAISS vector index. • Accept questions from the user. • Convert the question into a 384-dimensional embedding. • Search the FAISS index for semantically similar document chunks. • Retrieve the most relevant content and provide it as the answer. 3. Technologies Used Python Python is used as the primary programming language for implementing the complete system. PyPDF PyPDF is used to extract textual content from PDF documents. Sentence Tra